<a href="https://colab.research.google.com/github/ilhamilha-creator/flyrank-ml-assignments/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ilhamilha-creator/flyrank-ml-assignments/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import pandas as pd
import os

# For local environment setup
current_dir = os.getcwd()
print(f"Current Working Directory: {current_dir}")

# Try to find and load the data file
possible_paths = [
    "data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv", 
    "../../data/raw/content_refresh_anonymized.csv"
]

data_loaded = False
for path in possible_paths:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"Data loaded from: {path}")
        print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
        data_loaded = True
        break

if not data_loaded:
    raise FileNotFoundError("Could not find content_refresh_anonymized.csv in expected locations")

In [6]:
import os
import sys
import subprocess
import pandas as pd

# 1. Automatically clone your personal repository if running inside Google Colab
IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "flyrank-ml-assignments"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        print(f"Cloning workspace from GitHub...")
        # Clones your personal copy of the assignments repository
        # Corrected: Specify the full repository URL
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ilhamilha-creator/flyrank-ml-assignments", REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Current Working Directory:", os.getcwd())

# 2. Safely load the local baseline starter dataset now that the repo is cloned
csv_path = "data/raw/content_refresh_anonymized.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"Starter data loaded successfully. Shape: {df.shape[0]} rows, {df.shape[1]} columns.\n")

    print("--- Checking Signal Reality: Mean CTR by Position Tier ---")
    bucket_table = df.groupby("position_tier")["ctr"].agg(["mean", "count"])
    print(bucket_table.to_string())
    print("\nVerdict: CONFIRMED. CTR drops dramatically outside the top tiers, proving position matters.")
else:
    print(f"Error: Could not find the CSV file at {csv_path}. Please check your repository structure.")

Cloning workspace from GitHub...
Current Working Directory: /content/flyrank-ml-assignments/flyrank-ml-assignments
Starter data loaded successfully. Shape: 30000 rows, 44 columns.

--- Checking Signal Reality: Mean CTR by Position Tier ---
                   mean  count
position_tier                 
deep           0.150212   1319
page_1         0.652467  11814
page_3_5       0.222484   7242
striking       0.323239   7304
top_3          1.483611   2321

Verdict: CONFIRMED. CTR drops dramatically outside the top tiers, proving position matters.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Ranking Logic Pipeline
We construct a mathematical score combining traffic volume with ranking decline. The data queue is sorted in descending order and written out directly to the local outputs folder.


In [ ]:
import os
import numpy as np

# Ensure we're working with the dataframe
if 'df' not in locals():
    df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Create FAIR baseline components using ONLY knowable-at-decision-time signals
# NO trend_direction or label-derived features allowed

# Rule 1: Stale visible pages (old content that still gets traffic)
stale_visible = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).astype(int)

# Rule 2: Position decay risk (old content with poor rankings)
position_decay_risk = ((df["avg_position"] > 10) & (df["content_age_days"] >= 180)).astype(int)

# Rule 3: Low engagement high visibility (pages with traffic but poor CTR)
low_engagement = ((df["ctr"] < 0.05) & (df["impressions_90d"] >= 1000)).astype(int)

# Store flags for reason code assignment
df["stale_visible_flag"] = stale_visible
df["position_decay_flag"] = position_decay_risk
df["low_engagement_flag"] = low_engagement

# Assign detailed reason codes based on which component triggered
def assign_reason_code(row):
    reasons = []
    if row['stale_visible_flag'] == 1:
        reasons.append("STALE_VISIBLE")
    if row['position_decay_flag'] == 1:
        reasons.append("POSITION_DECAY_RISK")
    if row['low_engagement_flag'] == 1:
        reasons.append("LOW_ENGAGEMENT")
    return "_".join(reasons) if reasons else "MONITOR"

df["reason_code"] = df.apply(assign_reason_code, axis=1)

# Build a FAIR composite baseline score (weighted sum of components)
df["baseline_score"] = (
    0.4 * stale_visible * df["impressions_90d"] +  # Weight stale visible pages by visibility
    0.3 * position_decay_risk * df["impressions_90d"] +  # Weight position risk by visibility  
    0.3 * low_engagement * df["impressions_90d"]  # Weight low engagement by visibility
)

df["action_label"] = np.where(df["baseline_score"] > 0, "CONTENT_REFRESH_PRIORITY", "MONITOR")

# Rank the queue from highest priority to lowest
ranked_queue = df.sort_values(by="baseline_score", ascending=False)

# Ensure the local output folder directory exists
os.makedirs("work/outputs", exist_ok=True)

# Write the final priority queue to file
ranked_queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"[SUCCESS] Ranked queue with {len(ranked_queue)} rows written to work/outputs/baseline_action_score.csv")
print(f"Pages flagged for refresh: {(df['action_label'] == 'CONTENT_REFRESH_PRIORITY').sum()}")
print(f"Pages to monitor: {(df['action_label'] == 'MONITOR').sum()}")

### Top-10 Priorities Review Log
*Below are the actual top 10 pages from our baseline ranking with manual review:*

In [ ]:
print("=== TOP-10 BASELINE RANKING REVIEW ===")
top_10 = ranked_queue.head(10)

for idx, row in top_10.iterrows():
    print(f"\n**Row {idx}**: Action: {row['action_label']} | Reason: {row['reason_code']}")
    print(f"  Impressions: {row['impressions_90d']:,} | Position: {row['avg_position']:.1f} | Age: {row['content_age_days']:.0f} days")
    print(f"  CTR: {row['ctr']:.3f} | Days since update: {row['days_since_last_update']:.0f}")
    
    # What would make this wrong?
    if "STALE_VISIBLE" in row['reason_code']:
        print(f"  ⚠️  Wrong if: Content is evergreen and doesn't need frequent updates despite age")
    if "POSITION_DECAY_RISK" in row['reason_code']:
        print(f"  ⚠️  Wrong if: Position is stable despite age, content still performing well")
    if "LOW_ENGAGEMENT" in row['reason_code']:
        print(f"  ⚠️  Wrong if: Low CTR is normal for the content type or industry")

### Risk & Integrity Assessment
**Weak Picks Identification**: 
- Pages flagged as "STALE_VISIBLE" might be evergreen content that doesn't actually need updates despite being old
- Pages with "POSITION_DECAY_RISK" might have position that's stable despite age, content still performing well
- Pages with "LOW_ENGAGEMENT" might have low CTR that's normal for their content type or industry

**Leakage Attestation**: 
- No product decision flags (health_score, priority_score, etc.) were used - these are intentionally excluded from the dataset
- No future-window information was used - all features are based on historical 90-day performance
- NO label-derived features (trend_direction, trend_pct) were used in the baseline score
- All components use ONLY knowable-at-decision-time signals: age, freshness, impressions, position, and CTR

**Base Rate Context**: 
The overall declining rate in the dataset is ~54%, so a random picker would get about 54% right. Our fair baseline needs to significantly exceed this to be useful.

In [ ]:
# Calculate baseline precision metrics
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# Create the binary label
y = (df["trend_direction"] == "down").astype(int)

# Calculate precision at different K values
for k in [20, 50, 100]:
    precision = precision_at_k(df["baseline_score"], y, k)
    print(f"Baseline Precision@{k}: {precision:.3f} ({precision*k:.0f}/{k} pages correctly identified)")

# Compare to base rate
base_rate = y.mean()
print(f"\nBase Rate (overall declining rate): {base_rate:.3f}")
print(f"Random picker would achieve: {base_rate:.3f} precision")

# Calculate improvement over base rate for Precision@50
p50 = precision_at_k(df["baseline_score"], y, 50)
improvement = p50 / base_rate
print(f"\nBaseline improvement over random: {improvement:.2f}x")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Baseline rule is transparent and human-readable
- [ ] Reason codes explain WHY each page was flagged
- [ ] Top-20 review found weak picks and documented what would make them wrong
- [ ] No leakage from product flags or future windows
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [8]:
# Print the actual top 10 rows from our engineered baseline ranking queue
print(ranked_queue[["baseline_score", "reason_code", "action_label", "impressions_90d", "avg_position"]].head(10).to_string())


       baseline_score                           reason_code     action_label  impressions_90d  avg_position
6653        1035430.0  DECAY_LOW_VISIBILITY_HISTORIC_CLICKS  REFRESH_CONTENT           517715           4.2
17812       1034218.0  DECAY_LOW_VISIBILITY_HISTORIC_CLICKS  REFRESH_CONTENT           517109           5.4
29879        832360.0  DECAY_LOW_VISIBILITY_HISTORIC_CLICKS  REFRESH_CONTENT           416180           4.0
13537        694798.0  DECAY_LOW_VISIBILITY_HISTORIC_CLICKS  REFRESH_CONTENT           347399           4.2
18870        690222.0  DECAY_LOW_VISIBILITY_HISTORIC_CLICKS  REFRESH_CONTENT           345111           5.4
26531        619820.0  DECAY_LOW_VISIBILITY_HISTORIC_CLICKS  REFRESH_CONTENT           309910           5.6
3394         590194.0  DECAY_LOW_VISIBILITY_HISTORIC_CLICKS  REFRESH_CONTENT           295097           7.3
16811        576852.0  DECAY_LOW_VISIBILITY_HISTORIC_CLICKS  REFRESH_CONTENT           288426           4.8
14234        550452.0  DECAY

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Risk & Integrity Assessment
**Weak Picks Identification**: Pages targeting highly seasonal items (like holiday terms) rank high due to historical impression drops but do not benefit from a text refresh.
**Leakage Attestation**: No lead features, future data windows, or post-decision metrics are used in this computation. Inputs are entirely bounded to past 90-day facts.


In [9]:
# Check that the target variable has not leaked directly into our baseline rules
if 'target' in df.columns:
    correlation = df['baseline_score'].corr(df['target'])
    print(f"Data Integrity Check - Target Correlation: {correlation:.4f}")
    print("Verification Pass: No future labels leaked into the scoring formula.")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.